<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/09d_exercises.ipynb)

# 09d · 基础习题集：SELFIES、变分自编码器、代理模型防坑与扩散模型

本习题集配套第 09 讲（`09a` SELFIES 与分子 VAE、`09b` 目标导向逆向设计与代理模型防坑、`09c` 电镜图像超分辨率与去噪扩散模型），共包含 **8 道自包含的基础编程练习**，无需加载外部大型数据集即可独立运行检验。

---
## 习题 1：SELFIES 编码、符号切分与单点突变合法性验证

在 `09a` 中我们学习了 **SELFIES** 如何通过形式语法状态机保证 100% 的化学价态合法性。

请在下方实现函数 `selfies_roundtrip_and_mutate(smiles, replacement_token="[=O]")`：
1. 使用 `sf.encoder(smiles)` 将输入 SMILES 编码为 SELFIES 字符串，并用 `list(sf.split_selfies(...))` 切分为 token 列表；
2. 将 token 列表的正中间位置（`len(tokens) // 2`）替换为 `replacement_token`，再用 `sf.decoder("".join(tokens))` 解码回突变后的 SMILES；
3. 用 `Chem.MolFromSmiles` 检验突变后的 SMILES 是否依然是合法的分子，并返回 `(len(tokens), mutated_smiles, is_valid)`。

In [ ]:
import selfies as sf
from rdkit import Chem

def selfies_roundtrip_and_mutate(smiles, replacement_token="[=O]"):
    # TODO:
    # 1. 编码并切分为 SELFIES token 列表
    # 2. 将索引 len(tokens)//2 处的 token 替换为 replacement_token 并解码为 SMILES
    # 3. 返回 (原本 token 数, 突变后解码的 SMILES, Chem.MolFromSmiles 是否非 None)
    pass

# 运行自检
# print(selfies_roundtrip_and_mutate("c1ccc2c(c1)sc1c3ccccc3sc21"))

---
## 习题 2：手写 VAE 重参数化技巧与对角高斯 KL 散度

变分自编码器（VAE）的编码器输出均值 $\boldsymbol{\mu}$ 与对数方差 $\log \boldsymbol{\sigma}^2$（形状均为 `(B, d)`）：
1. **重参数化采样**：$\mathbf{z} = \boldsymbol{\mu} + \exp\!\left(\tfrac{1}{2}\log \boldsymbol{\sigma}^2\right) \odot \boldsymbol{\epsilon}$；
2. **批平均 KL 散度**：
   $$D_{\text{KL}} = -\frac{1}{2B}\sum_{i=1}^{B}\sum_{j=1}^{d}\left(1 + \log \sigma_{ij}^2 - \mu_{ij}^2 - \exp(\log \sigma_{ij}^2)\right)$$

请在下方补全 `reparameterize(mu, logvar, eps)` 与 `kl_divergence(mu, logvar)`，并验证当 `mu = 0, logvar = 0` 时 KL 散度严格等于 `0.0`。

In [ ]:
import torch

def reparameterize(mu, logvar, eps):
    # TODO: 使用给定的标准正态噪声张量 eps 计算 z = mu + sigma * eps
    pass

def kl_divergence(mu, logvar):
    # TODO: 返回 batch 平均的标量 KL 散度 (float 或 0D Tensor)
    pass

# 运行自检
# z_mu = torch.zeros(4, 16)
# z_lv = torch.zeros(4, 16)
# print("Zero KL check:", float(kl_divergence(z_mu, z_lv)))

---
## 习题 3：高维潜空间中的直线插值（Lerp）范数塌陷 vs. 球面线性插值（Slerp）

在 $d = 128$ 维潜空间中，两个正交且模长均为 $r$ 的向量 $\mathbf{z}_A, \mathbf{z}_B$ 的直线中点 $\frac{1}{2}(\mathbf{z}_A + \mathbf{z}_B)$ 模长会缩水为 $r / \sqrt{2} \approx 0.707\,r$，掉入高维球壳内部的空腔。

请在下方实现函数 `compare_midpoint_norms(z_a, z_b)`，计算 $t = 0.5$ 时的直线插值中点 `z_lerp_mid` 与球面线性插值中点 `z_slerp_mid`，并返回两者的欧氏范数 `(float(z_lerp_mid.norm()), float(z_slerp_mid.norm()))`。

In [ ]:
import torch

def slerp_midpoint(z_a, z_b):
    # TODO: 计算 t = 0.5 时的球面线性插值向量
    pass

def compare_midpoint_norms(z_a, z_b):
    # TODO: 返回 ( ||0.5*(z_a+z_b)||_2, ||slerp_midpoint(z_a, z_b)||_2 )
    pass

# 运行自检
# torch.manual_seed(0)
# a = torch.randn(128); a = a / a.norm() * 10.0
# b = torch.randn(128); b = b / b.norm() * 10.0
# print("Lerp vs Slerp midpoint norms:", compare_midpoint_norms(a, b))

---
## 习题 4：高维高斯球壳半径与 VAE 先验空洞（Prior Hole）定量计算

在 $d$ 维空间中，标准正态先验 $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}_d)$ 的欧氏范数平方 $\|\mathbf{z}\|_2^2 \sim \chi^2(d)$，其模长高度集中在 $\sqrt{d}$ 附近；若编码器学到的聚合后验每维标准差只有 $s < 1$（即 $\mathbf{z}_{\text{enc}} \sim \mathcal{N}(\mathbf{0}, s^2 \mathbf{I}_d)$），则真实样本居住在半径 $s\sqrt{d}$ 的内层球壳上。

请实现函数 `gaussian_shell_radii(d=128, posterior_std=0.55, n_samples=5000)`，通过蒙特卡洛采样返回 `(prior_median_norm, posterior_median_norm, theoretical_ratio)`。

In [ ]:
import torch

def gaussian_shell_radii(d=128, posterior_std=0.55, n_samples=5000):
    # TODO:
    # 1. 采样 z_prior ~ N(0, I) 与 z_post = posterior_std * z_prior
    # 2. 计算两者模长的中位数，以及它们的比值 posterior_median / prior_median
    pass

# 运行自检
# print(gaussian_shell_radii(128, 0.55))

---
## 习题 5：二值 Morgan 指纹的比特饱和（Bit Saturation）实验

在 `09b` 第 3.2 节中，我们发现随机森林代理模型给 6 聚体和 12 聚体打出了完全相同的分数，根源在于**二值 Morgan 指纹对重复局部子结构的饱和效应**。

请在下方编写函数 `count_oligomer_on_bits(monomer_smiles="c1ccc2nsnc2c1", lengths=(1, 2, 4, 6, 10))`：
1. 对每个聚合度 `k in lengths`，用 `"-".join([monomer_smiles] * k)` 构造 $k$ 聚体 SMILES；
2. 提取半径 `radius=2, fpSize=2048` 的二值 Morgan 指纹，统计其中取值为 `1` 的非零比特个数（`On-Bits`）；
3. 返回字典 `{k: on_bit_count}`，观察从 $k=3$ 以后非零比特数为何彻底停止增长。

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

def count_oligomer_on_bits(monomer_smiles="c1ccc2nsnc2c1", lengths=(1, 2, 3, 6, 10)):
    # TODO: 构造 k 聚体并统计 2048 位二值 Morgan 指纹中 1 的个数
    pass

# 运行自检
# print("On-bits vs oligomer length:", count_oligomer_on_bits())

---
## 习题 6：结合适用域过滤与悲观置信下界（LCB）的候选材料筛选器

给定 $M$ 个生成候选分子在包含 $T$ 棵决策树的随机森林上的预测矩阵 `tree_preds`（形状 `(M, T)`）以及每个候选分子到训练集的最近邻 Tanimoto 相似度数组 `max_sims`（形状 `(M,)`）：

请实现函数 `select_best_candidate_lcb(tree_preds, max_sims, sim_threshold=0.40, kappa=2.0)`：
1. 计算每个候选分子的预测均值 $\hat{\mu}_i$、树间标准差 $\hat{\sigma}_i$ 以及悲观得分 $\text{LCB}_i = \hat{\mu}_i - \kappa\,\hat{\sigma}_i$；
2. 过滤掉所有 `max_sims < sim_threshold` 的分布外分子；
3. 在通过适用域过滤的候选分子中，返回具有最高 $\text{LCB}_i$ 的原始候选索引 `best_idx` 与其对应的 `float(LCB[best_idx])`。

In [ ]:
import numpy as np

def select_best_candidate_lcb(tree_preds, max_sims, sim_threshold=0.40, kappa=2.0):
    # TODO: 过滤 max_sims < sim_threshold 的样本，在剩余样本中按 mean - kappa*std 选出最优索引与得分
    pass

# 运行自检
# fake_preds = np.array([[8.0, 4.0, 9.0, 3.0], [6.5, 6.4, 6.6, 6.5], [7.2, 7.0, 7.1, 7.3]])
# fake_sims  = np.array([0.55, 0.62, 0.25])  # 第 2 号虽然均值高但超出适用域 (0.25 < 0.40)
# print("Selected (idx, LCB):", select_best_candidate_lcb(fake_preds, fake_sims))

---
## 习题 7：电镜超分辨率图像的峰值信噪比（PSNR）计算

在 `09c` 中，峰值信噪比（PSNR）定义为（假设像素值归一化在 $[0, 1]$ 区间）：

$$\text{MSE} = \frac{1}{N}\sum_{k=1}^{N} (\hat{I}_k - I_k)^2, \qquad \text{PSNR (dB)} = -10 \log_{10}(\text{MSE})$$

请实现函数 `compute_psnr_db(pred, target)`，并验证当均方误差缩小为原来的 $1/4$（即噪声幅度减半）时，PSNR 提升了约 $6.02\text{ dB}$。

In [ ]:
import math
import torch

def compute_psnr_db(pred, target):
    # TODO: 计算 MSE 并返回 float(-10 * log10(MSE))
    pass

# 运行自检
# gt = torch.ones(1, 1, 16, 16) * 0.5
# p1 = gt + 0.10
# p2 = gt + 0.05
# print(f"PSNR(err=0.10) = {compute_psnr_db(p1, gt):.2f} dB, PSNR(err=0.05) = {compute_psnr_db(p2, gt):.2f} dB")

---
## 习题 8：去噪扩散模型（DDPM）的前向闭合加噪与信噪比计算

给定线性方差调度 $\boldsymbol{\beta} = (\beta_1, \dots, \beta_T)$，令 $\alpha_t = 1 - \beta_t$，$\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$。

请在下方实现函数 `ddpm_forward_step(x0, t_index, betas, noise)`：
1. 计算累积信号保留率序列 `alphas_bar = torch.cumprod(1.0 - betas, dim=0)`；
2. 取出标量下标 `t_index` 对应的 $\bar{\alpha}_t$，计算加噪图像 $\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon}$；
3. 计算该时刻的信噪比 $\text{SNR}(t) = \bar{\alpha}_t / (1 - \bar{\alpha}_t)$，返回 `(x_t, float(snr_t))`。

In [ ]:
import torch

def ddpm_forward_step(x0, t_index, betas, noise):
    # TODO: 计算 alphas_bar[t_index]，返回 (x_t, float(snr))
    pass

# 运行自检
# betas_demo = torch.linspace(1e-4, 0.08, 40)
# x0_demo = torch.ones(1, 1, 8, 8)
# eps_demo = torch.zeros(1, 1, 8, 8)
# xt_out, snr_val = ddpm_forward_step(x0_demo, 10, betas_demo, eps_demo)
# print(f"x_t mean: {xt_out.mean().item():.4f}, SNR(t=10): {snr_val:.4f}")